In [24]:
import pandas as pd
import numpy as np

import gspread
import panel as pn
pn.extension('tabulator')
import hvplot.pandas
import holoviews as hv
hv.extension('bokeh')
pn.extension(theme='dark')

In [25]:
#after renaming the json file
gc = gspread.service_account(filename="service_account.json")
sh = gc.open('revolut')
ws=sh.worksheet('Sheet1')
df = pd.DataFrame(ws.get_all_records()) 
df.head()


,Date,Narration,Chq./Ref.No.,Value Dt,Amount,Closing Balance
0,02/03/25,UPI-AISHWARYA G,100805656166,02/03/25,-60,20786.22
1,02/03/25,UPI-AISHWARYA G,506146225981,02/03/25,60,20846.22
2,03/03/25,UPI-AISHWARYA G,542850689083,03/03/25,20,20866.22
3,03/03/25,UPI-SAVITHA A C,100869600911,03/03/25,-40,20826.22
4,03/03/25,UPI-SYED RAHILL PASHA,100871069345,03/03/25,-77,20749.22


In [26]:
#cleaning the dataframe
df = df[['Date', 'Narration', 'Amount']]
#df['Narration']=df['Narration'].map(str.lower)
df = df.rename(columns={'Narration':'Description'})
#add a category column
df['Category']='unassigned'
df.head()

,Date,Description,Amount,Category
0,02/03/25,UPI-AISHWARYA G,-60,unassigned
1,02/03/25,UPI-AISHWARYA G,60,unassigned
2,03/03/25,UPI-AISHWARYA G,20,unassigned
3,03/03/25,UPI-SAVITHA A C,-40,unassigned
4,03/03/25,UPI-SYED RAHILL PASHA,-77,unassigned


In [27]:
""" 
Categories defined as:
1. Friend
2. Transport
3. Self Care
4. Grocery
5. Dine out
"""

' \nCategories defined as:\n1. Friend\n2. Transport\n3. Self Care\n4. Grocery\n5. Dine out\n'

In [28]:
df['Category'] = np.where(df['Description'].str.contains('UPI-AISHWARYA  G'),'Friend',df['Category'])
df['Category'] = np.where(df['Description'].str.contains('UPI-SYED RAHILL PASHA'),'Transport',df['Category'])
df['Category'] = np.where(df['Description'].str.contains('UPI-SARALA LAMA'),'Self Care',df['Category'])
df['Category'] = np.where(df['Description'].str.contains('UPI-VAIDIC DHARMA SANSTH|UPI-SAVITHA A C|UPI-SRI MOOKAMBIKA HOSP|UPI-ADITISHARMA'),'Dine Out',df['Category'])
df['Category'] = np.where(df['Description'].str.contains('UPI-TARAKARI MANDI'),'Grocery',df['Category'])

#set date to datetime format
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year
pd.options.display.max_rows=999
df.head(200)


C:\Users\Anukriti Sharma\AppData\Local\Temp\ipykernel_14140\2573327254.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Date'] = pd.to_datetime(df['Date'])


,Date,Description,Amount,Category,Month,Year
0,2025-02-03,UPI-AISHWARYA G,-60,Friend,2,2025
1,2025-02-03,UPI-AISHWARYA G,60,Friend,2,2025
2,2025-03-03,UPI-AISHWARYA G,20,Friend,3,2025
3,2025-03-03,UPI-SAVITHA A C,-40,Dine Out,3,2025
4,2025-03-03,UPI-SYED RAHILL PASHA,-77,Transport,3,2025
5,2025-06-03,UPI-SARALA LAMA,-600,Self Care,6,2025
6,2025-11-03,UPI-AISHWARYA G,-20,Friend,11,2025
7,2025-03-16,UPI-VAIDIC DHARMA SANSTH,-20,Dine Out,3,2025
8,2025-03-16,UPI-AISHWARYA G,-75,Friend,3,2025
9,2025-03-16,UPI-SRI MOOKAMBIKA HOSP,-50,Dine Out,3,2025


In [29]:
#find the transations not assigned a category yet
unassigned = df.loc[df['Category']=='unassigned']
unassigned


,Date,Description,Amount,Category,Month,Year
15,2025-01-04,INTEREST PAID TILL 31-MAR-2025,173,unassigned,1,2025


In [30]:
latest_month=df['Month'].max()
latest_year=df['Year'].max()
#filter the df to hold only recent expenses

latest_expense=df[((df['Month']==latest_month)|(df['Month']==latest_month-1))&(df['Year']==latest_year)]

In [31]:
#FOR ELEMENT 1
last_month_expenses=latest_expense.groupby('Category')['Amount'].sum().reset_index()
last_month_expenses['Amount']=last_month_expenses['Amount'].astype('str')
last_month_expenses['Amount']=last_month_expenses['Amount'].str.replace('-',' ')
last_month_expenses['Amount']=last_month_expenses['Amount'].astype('float')
#after getting the absolute figures, get all those which have assigned categories

last_month_expenses=last_month_expenses[last_month_expenses['Category'].str.contains('unassigned')==False]
last_month_expenses=last_month_expenses.sort_values(by='Amount', ascending=False)
last_month_expenses['Amount']=last_month_expenses['Amount'].round().astype(int)

last_month_expenses

,Category,Amount
0,Friend,20


In [32]:
last_month_expenses_tot=last_month_expenses['Amount'].sum()
float(last_month_expenses_tot)

20.0

In [33]:
def calc_diff(event):
    income=float(income_widget.value)
    recurring_expenses=float(recurring_expenses_widget.value)
    monthly_expenses=float(monthly_expenses_widget.value)
    diff=income-recurring_expenses-monthly_expenses
    difference_widget.value=str(diff)

income_widget=pn.widgets.TextInput(name='Income',value="0")
recurring_expenses_widget=pn.widgets.TextInput(name='Recurring Expenses',value='0')
monthly_expenses_widget=pn.widgets.TextInput(name='Non-Recurring Expenses', value=str(last_month_expenses_tot))
difference_widget=pn.widgets.TextInput(name="Last Month's Savings", value="0")

income_widget.param.watch(calc_diff,"value")
recurring_expenses_widget.param.watch(calc_diff,"value")
monthly_expenses_widget.param.watch(calc_diff,"value")
pn.Row(income_widget,recurring_expenses_widget,monthly_expenses_widget,difference_widget).show()


Launching server at http://localhost:59689


In [34]:
#creating the bar chart to visualize all the categories
last_month_expenses_chart=last_month_expenses.hvplot.bar(
    x='Category',
    y='Amount',
    height=250,
    width=850,
    title='Last Month Expenses',
    ylim=(0,500)
)
last_month_expenses_chart



:Bars   [Category]   (Amount)

In [35]:
#monthly expenses trend bar chart
df['Date']=pd.to_datetime(df['Date'])
df['Month-Year']=df['Date'].dt.to_period('M')
monthly_expenses_by_cat =df.groupby(['Month-Year','Category'])['Amount'].sum().reset_index()

monthly_expenses_by_cat['Amount']=monthly_expenses_by_cat['Amount'].astype('str')
monthly_expenses_by_cat['Amount']=monthly_expenses_by_cat['Amount'].str.replace('-',' ')
monthly_expenses_by_cat['Amount']=monthly_expenses_by_cat['Amount'].astype('float')
monthly_expenses_by_cat=monthly_expenses_by_cat[monthly_expenses_by_cat['Category'].str.contains("unassigned")==False]
monthly_expenses_by_cat=monthly_expenses_by_cat.sort_values(by='Amount', ascending=False)
monthly_expenses_by_cat['Amount']=monthly_expenses_by_cat['Amount'].round().astype(int)
monthly_expenses_by_cat['Month-Year']=monthly_expenses_by_cat['Month-Year'].astype('str')
monthly_expenses_by_cat=monthly_expenses_by_cat.rename(columns={'Amount': 'Amount '})
monthly_expenses_by_cat


,Month-Year,Category,Amount
7,2025-06,Self Care,600
3,2025-03,Dine Out,270
6,2025-03,Transport,77
5,2025-03,Grocery,70
8,2025-11,Friend,20
4,2025-03,Friend,15
0,2025-01,Friend,10
2,2025-02,Friend,0


In [36]:

    #let's define the panel widget
select_cat1=pn.widgets.Select(name='Select Category', options=['All']+list(monthly_expenses_by_cat['Category'].unique()))
select_cat1

BokehModel(combine_events=True, render_bundle={'docs_json': {'83969e14-5845-4285-bc6f-6906341afeaf': {'version…

In [37]:
def plot_expenses(category):
    if category=='All':
        plot_df=monthly_expenses_by_cat.groupby('Month-Year').sum()
    else:
        plot_df=monthly_expenses_by_cat[monthly_expenses_by_cat['Category']==category].groupby('Month-Year').sum()
    plot=plot_df.hvplot.bar(x='Month-Year', y='Amount ')
    return plot

#define a callback for when the value changes after you pick something else
@pn.depends(select_cat1.param.value)
def update_plot(category):
    plot=plot_expenses(category)
    return plot
monthly_expenses_by_cat_chart=pn.Row(select_cat1, update_plot)
monthly_expenses_by_cat_chart[1].width=600
monthly_expenses_by_cat_chart

BokehModel(combine_events=True, render_bundle={'docs_json': {'178a8dec-84c3-4b5f-8699-a83c3a6abcb0': {'version…

In [38]:
#summary table

df=df[['Date','Category','Description','Amount']]
df['Amount']=df['Amount'].astype('str')
df['Amount']=df['Amount'].str.replace('-',' ')
df['Amount']=df['Amount'].astype('float')
df=df[df['Category'].str.contains('unassigned')==False]
df['Amount']=df['Amount'].round().astype(int)
df

,Date,Category,Description,Amount
0,2025-02-03,Friend,UPI-AISHWARYA G,60
1,2025-02-03,Friend,UPI-AISHWARYA G,60
2,2025-03-03,Friend,UPI-AISHWARYA G,20
3,2025-03-03,Dine Out,UPI-SAVITHA A C,40
4,2025-03-03,Transport,UPI-SYED RAHILL PASHA,77
5,2025-06-03,Self Care,UPI-SARALA LAMA,600
6,2025-11-03,Friend,UPI-AISHWARYA G,20
7,2025-03-16,Dine Out,UPI-VAIDIC DHARMA SANSTH,20
8,2025-03-16,Friend,UPI-AISHWARYA G,75
9,2025-03-16,Dine Out,UPI-SRI MOOKAMBIKA HOSP,50


In [39]:
#filter df based on category
def filter_df(category):
    if category=='All':
        return df
    return df[df['Category']==category]

summary_table=pn.widgets.DataFrame(filter_df('All'), height=500, width=650)
def update_summary(event):
    summary_table.value=filter_df(event.new)
select_cat1.param.watch(update_summary, 'value')
summary_table

BokehModel(combine_events=True, render_bundle={'docs_json': {'7c19efcd-98d5-42dd-8866-ab3de273acb1': {'version…

In [ ]:
#finally put everything into our dashboard
template = pn.template.FastListTemplate(
    title = "Anukriti's Personal Finances Summary",
    sidebar=[
        pn.pane.Markdown("*Don't go broke trying to look rich...*"),
        pn.pane.PNG('money.png', sizing_mode='scale_both'),
        pn.pane.Markdown(" "),
        pn.pane.Markdown(" "), 
        select_cat1
    ],
    main=[
        
        pn.Row(income_widget, recurring_expenses_widget, monthly_expenses_widget, difference_widget, width=950),
        pn.Row(last_month_expenses_chart, height=240),
        pn.GridBox(
            monthly_expenses_by_cat_chart[1],
            ncols=2,
            width=400,
            height=400,
            align='end',
            sizing_mode='stretch_width'
        ),
        pn.GridBox(summary_table, sizing_mode='scale_width')

    ]
    
)

template.show()

Launching server at http://localhost:59697
